## Example 1: Basic fmin Optimization with hp Search Spaces

Demonstrates the core fmin function and hp search space definitions unique to hyperopt.

In [ ]:
# Example 1: Basic fmin optimization with hp search spaces
try:
    from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
    import hyperopt
    import numpy as np
    
    print("=== Hyperopt Package Overview ===")
    print(f"Hyperopt version: {hyperopt.__version__ if hasattr(hyperopt, '__version__') else 'unknown'}")
    
    # Available optimization algorithms
    print(f"Available algorithms: TPE (Tree-structured Parzen Estimator), Random, Anneal")
    
    print("\n=== Define Search Space with hp Functions ===")
    # Demonstrate unique hp search space functions
    space = {
        'learning_rate': hp.uniform('learning_rate', 0.001, 0.1),  # Uniform distribution
        'n_estimators': hp.choice('n_estimators', [50, 100, 200, 300]),  # Discrete choice
        'max_depth': hp.quniform('max_depth', 3, 10, 1),  # Quantized uniform
        'regularization': hp.loguniform('regularization', np.log(1e-4), np.log(1e-1))  # Log-uniform
    }
    
    print(f"Search space defined with {len(space)} parameters using hp functions:")
    print(f"  hp.uniform: continuous uniform [0.001, 0.1]")
    print(f"  hp.choice: discrete choice from list")
    print(f"  hp.quniform: quantized uniform (integers)")
    print(f"  hp.loguniform: log-scale uniform distribution")
    
    print("\n=== fmin Optimization ===")
    # Define objective function to minimize
    def objective(params):
        # Simulate model performance (lower is better)
        lr = params['learning_rate']
        n_est = params['n_estimators']
        depth = params['max_depth']
        reg = params['regularization']
        
        # Simulate validation loss
        loss = 0.5 + lr * 0.1 + (n_est - 150) ** 2 / 10000 + depth * 0.02 + reg * 0.1
        return {'loss': loss, 'status': STATUS_OK}
    
    # Run fmin optimization with TPE algorithm
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,  # Use Tree of Parzen Estimators
        max_evals=10,
        trials=trials,
        verbose=0
    )
    
    print(f"Best parameters found by fmin: {best}")
    print(f"Number of trials executed: {len(trials)}")
    print(f"Best loss achieved: {min([t['result']['loss'] for t in trials.trials]):.4f}")
    
    from hyperopt import space_eval
    losses = [t['result']['loss'] for t in trials.trials]
    assert len(trials) == 10
    assert set(best) == {'learning_rate', 'n_estimators', 'max_depth', 'regularization'}
    # fmin reports hp.choice parameters as indexes into the option list
    assert best['n_estimators'] in range(4)
    assert 0.001 <= best['learning_rate'] <= 0.1
    assert 3 <= best['max_depth'] <= 10 and best['max_depth'] == int(best['max_depth'])
    assert 1e-4 <= best['regularization'] <= 1e-1
    # The best point is the trial with the lowest loss, and re-evaluating it gives the same loss
    assert trials.best_trial['result']['loss'] == min(losses)
    assert np.isclose(objective(space_eval(space, best))['loss'], min(losses))
    
except Exception as e:
    print(f"Example 1: Error in basic optimization test: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 2: Advanced Search Spaces and Conditional Parameters

Demonstrate complex search spaces with conditional parameters and nested structures.

In [ ]:
# Example 2: Advanced search spaces and conditional parameters
try:
    from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
    import numpy as np
    
    print("=== Complex Search Space Definition ===")
    # Define conditional and nested search space
    space = hp.choice('model_type', [
        {
            'type': 'random_forest',
            'n_estimators': hp.choice('rf_n_estimators', [50, 100, 200]),
            'max_depth': hp.randint('rf_max_depth', 3, 15),
            'min_samples_split': hp.uniform('rf_min_samples_split', 0.1, 0.5)
        },
        {
            'type': 'gradient_boosting',
            'n_estimators': hp.choice('gb_n_estimators', [100, 200, 500]),
            'learning_rate': hp.uniform('gb_learning_rate', 0.01, 0.3),
            'max_depth': hp.randint('gb_max_depth', 3, 8)
        },
        {
            'type': 'svm',
            'C': hp.lognormal('svm_C', 0, 2),
            'kernel': hp.choice('svm_kernel', ['rbf', 'linear', 'poly']),
            'gamma': hp.choice('svm_gamma', ['scale', 'auto'])
        }
    ])
    
    print("Conditional search space with 3 model types:")
    print("- Random Forest with 4 hyperparameters")
    print("- Gradient Boosting with 3 hyperparameters")
    print("- SVM with 3 hyperparameters")
    
    print("\n=== Model-Specific Objective Function ===")
    def objective(params):
        model_type = params['type']
        
        if model_type == 'random_forest':
            # Simulate Random Forest performance
            base_score = 0.85
            n_est = params['n_estimators']
            depth = params['max_depth']
            min_split = params['min_samples_split']
            
            # Simple performance simulation
            performance = base_score + n_est / 1000 - (depth - 8) ** 2 / 100 + min_split * 0.1
            
        elif model_type == 'gradient_boosting':
            # Simulate Gradient Boosting performance
            base_score = 0.87
            n_est = params['n_estimators']
            lr = params['learning_rate']
            depth = params['max_depth']
            
            performance = base_score + n_est / 2000 + lr * 0.5 - (depth - 5) ** 2 / 50
            
        else:  # SVM
            base_score = 0.82
            C = params['C']
            kernel = params['kernel']
            
            kernel_bonus = {'rbf': 0.05, 'linear': 0.02, 'poly': 0.01}[kernel]
            performance = base_score + np.log(C) * 0.02 + kernel_bonus
        
        # Convert to loss (minimize)
        loss = 1.0 - performance
        
        return {'loss': loss, 'status': STATUS_OK, 'model_type': model_type}
    
    # Run optimization
    trials = Trials()
    best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=15, trials=trials)
    
    print(f"Best configuration: {best}")
    
    # Analyze results by model type
    model_types = [trial['result']['model_type'] for trial in trials.trials]
    print(f"Models tried: {set(model_types)}")
    
    from hyperopt import space_eval
    assert len(trials) == 15
    assert set(model_types) <= {'random_forest', 'gradient_boosting', 'svm'}
    # space_eval maps the best indexes back to the chosen branch of the conditional space
    best_config = space_eval(space, best)
    assert best_config['type'] == ['random_forest', 'gradient_boosting', 'svm'][best['model_type']]
    assert trials.best_trial['result']['model_type'] == best_config['type']
    
except Exception as e:
    print(f"Example 2: Error in advanced search test: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 3: Trials Management and Result Analysis

Demonstrate trial tracking, result analysis, and optimization progress monitoring.

In [ ]:
# Example 3: Trials management and result analysis
try:
    from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
    import numpy as np
    
    print("=== Trials Object and Tracking ===")
    # Create trials object to track optimization
    trials = Trials()
    
    # Define simple search space
    space = {
        'x': hp.uniform('x', -5, 5),
        'y': hp.normal('y', 0, 2)
    }
    
    # Objective function with additional metrics
    def objective_with_metrics(params):
        x, y = params['x'], params['y']
        
        # Complex function with multiple minima
        loss = (x - 2) ** 2 + (y + 1) ** 2 + 0.1 * np.sin(5 * x) + 0.1 * np.cos(5 * y)
        
        # Additional metrics to track
        distance_from_origin = np.sqrt(x ** 2 + y ** 2)
        
        return {
            'loss': loss,
            'status': STATUS_OK,
            'eval_time': np.random.uniform(0.1, 1.0),  # Simulate evaluation time
            'distance': distance_from_origin,
            'x_val': x,
            'y_val': y
        }
    
    # Run optimization with trial tracking
    best = fmin(
        fn=objective_with_metrics,
        space=space,
        algo=tpe.suggest,
        max_evals=20,
        trials=trials
    )
    
    print(f"Best parameters: {best}")
    print(f"Total trials completed: {len(trials)}")
    
    print("\n=== Trial Analysis ===")
    # Extract results from trials
    losses = [trial['result']['loss'] for trial in trials.trials]
    distances = [trial['result']['distance'] for trial in trials.trials]
    eval_times = [trial['result']['eval_time'] for trial in trials.trials]
    
    print(f"Best loss: {min(losses):.4f}")
    print(f"Average loss: {np.mean(losses):.4f}")
    print(f"Loss std: {np.std(losses):.4f}")
    print(f"Average eval time: {np.mean(eval_times):.3f}s")
    
    # Find best trial details
    best_trial_idx = np.argmin(losses)
    best_trial = trials.trials[best_trial_idx]
    print(f"Best trial result: {best_trial['result']}")
    
    print("\n=== Space Evaluation ===")
    # Convert best hyperopt format back to space format
    best_space = space_eval(space, best)
    print(f"Best parameters in original space: {best_space}")
    
    # Trial state information
    print(f"Trial states: {[trial['state'] for trial in trials.trials[:3]]}")
    
    from hyperopt import JOB_STATE_DONE
    assert len(trials) == 20
    assert all(trial['state'] == JOB_STATE_DONE for trial in trials.trials)
    assert -5 <= best['x'] <= 5
    assert best_trial['result']['loss'] == min(losses) == trials.best_trial['result']['loss']
    # Extra keys returned by the objective are stored with each trial
    assert np.isclose(best_trial['result']['distance'], np.hypot(best_trial['result']['x_val'], best_trial['result']['y_val']))
    assert best_space == {'x': best['x'], 'y': best['y']}
    
except Exception as e:
    print(f"Example 3: Error in trials management test: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 4: Different Optimization Algorithms

Compare different optimization algorithms and their performance characteristics.

In [ ]:
# Example 4: Different optimization algorithms comparison
try:
    from hyperopt import fmin, tpe, rand, anneal, hp, Trials, STATUS_OK
    import numpy as np
    
    print("=== Optimization Algorithm Comparison ===")
    # Define a challenging objective function
    def challenging_objective(params):
        x = params['x']
        y = params['y']
        
        # Multi-modal function with noise
        result = np.sin(x) * np.cos(y) + 0.1 * (x - 1) ** 2 + 0.1 * (y + 0.5) ** 2
        result += 0.05 * np.random.normal()  # Add noise
        
        return {'loss': result, 'status': STATUS_OK}
    
    # Search space
    space = {
        'x': hp.uniform('x', -5, 5),
        'y': hp.uniform('y', -5, 5)
    }
    
    # Test different algorithms
    algorithms = {
        'TPE': tpe.suggest,
        'Random Search': rand.suggest,
        'Simulated Annealing': anneal.suggest
    }
    
    results = {}
    
    for algo_name, algo_func in algorithms.items():
        print(f"\n--- Testing {algo_name} ---")
        
        trials = Trials()
        
        best = fmin(
            fn=challenging_objective,
            space=space,
            algo=algo_func,
            max_evals=15,
            trials=trials
        )
        
        # Analyze results
        losses = [trial['result']['loss'] for trial in trials.trials]
        best_loss = min(losses)
        avg_loss = np.mean(losses)
        
        results[algo_name] = {
            'best_params': best,
            'best_loss': best_loss,
            'avg_loss': avg_loss,
            'losses': losses
        }
        
        print(f"Best loss: {best_loss:.4f}")
        print(f"Average loss: {avg_loss:.4f}")
        print(f"Best params: x={best['x']:.3f}, y={best['y']:.3f}")
    
    print("\n=== Algorithm Performance Summary ===")
    for algo_name, result in results.items():
        print(f"{algo_name:20}: Best={result['best_loss']:.4f}, Avg={result['avg_loss']:.4f}")
    
    # Find overall best algorithm
    best_algo = min(results.keys(), key=lambda k: results[k]['best_loss'])
    print(f"Best performing algorithm: {best_algo}")
    
    print("\n=== Convergence Analysis ===")
    # Analyze convergence patterns
    for algo_name, result in results.items():
        losses = result['losses']
        # Calculate running minimum
        running_min = [min(losses[:i+1]) for i in range(len(losses))]
        improvement = running_min[0] - running_min[-1]
        print(f"{algo_name} improvement: {improvement:.4f}")
    
    for algo_name, result in results.items():
        assert len(result['losses']) == 15, algo_name
        assert result['best_loss'] == min(result['losses'])
        assert -5 <= result['best_params']['x'] <= 5 and -5 <= result['best_params']['y'] <= 5
        running_min = [min(result['losses'][:i + 1]) for i in range(len(result['losses']))]
        # The best-so-far loss never gets worse
        assert all(a >= b for a, b in zip(running_min, running_min[1:]))
    
except Exception as e:
    print(f"Example 4: Error in algorithm comparison test: {e}")
    import traceback
    traceback.print_exc()
    raise

## Example 5: Production Hyperparameter Tuning Pipeline

Demonstrate a complete hyperparameter tuning pipeline with early stopping and cross-validation.

In [ ]:
# Example 5: Production hyperparameter tuning pipeline
try:
    from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, STATUS_FAIL
    import numpy as np
    import time
    
    print("=== Production Pipeline Setup ===")
    # Simulate a complete ML pipeline
    class MLPipeline:
        def __init__(self):
            self.best_score = float('inf')
            self.early_stop_patience = 5
            self.no_improve_count = 0
        
        def cross_validate(self, params):
            """Simulate cross-validation"""
            # Simulate different CV folds
            fold_scores = []
            
            for fold in range(3):  # 3-fold CV
                # Simulate training with these parameters
                lr = params['learning_rate']
                l2 = params['l2_reg']
                dropout = params['dropout']
                
                # Add some randomness per fold
                fold_variance = np.random.normal(0, 0.02)
                score = 0.15 + lr * 2 + l2 * 0.5 + dropout * 0.3 + fold_variance
                fold_scores.append(score)
            
            # Return mean and std of CV scores
            mean_score = np.mean(fold_scores)
            std_score = np.std(fold_scores)
            
            return mean_score, std_score
    
    pipeline = MLPipeline()
    
    print("Pipeline created with 3-fold cross-validation")
    
    print("\n=== Advanced Search Space ===")
    # Define comprehensive search space
    space = {
        'learning_rate': hp.loguniform('learning_rate', 
                                     low=np.log(0.001), high=np.log(0.1)),
        'l2_reg': hp.loguniform('l2_reg', 
                               low=np.log(1e-6), high=np.log(1e-2)),
        'dropout': hp.uniform('dropout', 0.0, 0.5),
        'batch_size': hp.choice('batch_size', [16, 32, 64, 128]),
        'optimizer': hp.choice('optimizer', ['adam', 'sgd', 'rmsprop'])
    }
    
    print("Search space includes:")
    print("- Learning rate: log-uniform [0.001, 0.1]")
    print("- L2 regularization: log-uniform [1e-6, 1e-2]") 
    print("- Dropout: uniform [0.0, 0.5]")
    print("- Batch size: choice [16, 32, 64, 128]")
    print("- Optimizer: choice ['adam', 'sgd', 'rmsprop']")
    
    print("\n=== Objective Function with Error Handling ===")
    trial_counter = {'count': 0}
    
    def objective(params):
        trial_counter['count'] += 1
        trial_id = trial_counter['count']
        
        try:
            # Simulate parameter validation
            if params['learning_rate'] > 0.05 and params['dropout'] < 0.1:
                # Simulate parameter conflict
                return {'status': STATUS_FAIL, 
                       'failure_reason': 'Parameter conflict: high LR + low dropout'}
            
            # Run cross-validation
            mean_score, std_score = pipeline.cross_validate(params)
            
            # Early stopping logic
            if mean_score < pipeline.best_score:
                pipeline.best_score = mean_score
                pipeline.no_improve_count = 0
            else:
                pipeline.no_improve_count += 1
            
            # Additional metrics
            optimizer_penalty = {'adam': 0, 'sgd': 0.01, 'rmsprop': 0.005}
            final_score = mean_score + optimizer_penalty[params['optimizer']]
            
            return {
                'loss': final_score,
                'status': STATUS_OK,
                'cv_mean': mean_score,
                'cv_std': std_score,
                'trial_id': trial_id,
                'eval_time': np.random.uniform(10, 60)  # Simulate eval time
            }
            
        except Exception as e:
            return {
                'status': STATUS_FAIL,
                'failure_reason': str(e),
                'trial_id': trial_id
            }
    
    # Run optimization with comprehensive tracking
    trials = Trials()
    
    print("Starting hyperparameter optimization...")
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=25,
        trials=trials
    )
    
    print(f"\nOptimization completed!")
    print(f"Best parameters: {best}")
    
    # Analyze results
    successful_trials = [t for t in trials.trials if t['result']['status'] == STATUS_OK]
    failed_trials = [t for t in trials.trials if t['result']['status'] == STATUS_FAIL]
    
    print(f"\nResults summary:")
    print(f"Successful trials: {len(successful_trials)}")
    print(f"Failed trials: {len(failed_trials)}")
    
    if successful_trials:
        best_result = min(successful_trials, key=lambda t: t['result']['loss'])
        print(f"Best CV score: {best_result['result']['cv_mean']:.4f} ± {best_result['result']['cv_std']:.4f}")
        print(f"Best trial evaluation time: {best_result['result']['eval_time']:.1f}s")
    
    from hyperopt import space_eval
    assert len(trials.trials) == 25 == trial_counter['count']
    assert len(successful_trials) + len(failed_trials) == 25
    # Every failed trial is one the objective rejected: learning rate > 0.05 with dropout < 0.1
    for t in failed_trials:
        vals = t['misc']['vals']
        assert vals['learning_rate'][0] > 0.05 and vals['dropout'][0] < 0.1
        assert t['result']['failure_reason'].startswith('Parameter conflict')
    # fmin only picks the best point among successful trials
    best_params = space_eval(space, best)
    assert not (best_params['learning_rate'] > 0.05 and best_params['dropout'] < 0.1)
    assert best_params['optimizer'] in ('adam', 'sgd', 'rmsprop')
    
except Exception as e:
    print(f"Example 5: Error in production pipeline test: {e}")
    import traceback
    traceback.print_exc()
    raise